In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [62]:
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install opencv-contrib-python>=4.7.0.72

In [56]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow


# Ruta de tu imagen (CAMBIA ESTO)
image_path = "/content/drive/MyDrive/Robotica 1/prueba1.jpg"


In [76]:
import cv2
import numpy as np

def estimate_pose_SOLVEPNP(corners, marker_size, K, dist):
    half = marker_size / 2.0

    obj_points = np.array([
        [-half,  half, 0],
        [ half,  half, 0],
        [ half, -half, 0],
        [-half, -half, 0]
    ], dtype=np.float32)

    img_points = corners.reshape((4,2)).astype(np.float32)

    ok, rvec, tvec = cv2.solvePnP(
        obj_points, img_points, K, dist,
        flags=cv2.SOLVEPNP_IPPE_SQUARE
    )

    return rvec, tvec


In [78]:
def detectar_area_aruco(frame, marker_size_cm=2.7, origin_id=None):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    aruco = cv2.aruco
    aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)

    params = aruco.DetectorParameters()

    # ✅ API NUEVA (tu versión)
    detector = cv2.aruco.ArucoDetector(aruco_dict, params)

    corners, ids, _ = detector.detectMarkers(gray)

    if ids is None:
        return None, None, None, "No se detectaron marcadores"

    ids = ids.flatten()
    aruco.drawDetectedMarkers(frame, corners, ids.reshape(-1, 1))

    # ✅ Cámara ficticia
    K = np.array([[800, 0, 320],
                  [0, 800, 240],
                  [0,   0,   1]], float)
    dist = np.zeros(5)

    # ✅ obtener poses
    poses = {}
    for i, marker_id in enumerate(ids):
        rvec, tvec = estimate_pose_SOLVEPNP(
            corners[i][0], marker_size_cm, K, dist
        )
        poses[marker_id] = (rvec, tvec.reshape(3,1))

    # ✅ origen
    if origin_id is None:
        origin_id = ids[0]

    if origin_id not in poses:
        return None, None, None, f"No está el marcador origen {origin_id}"

    r0, t0 = poses[origin_id]
    R0, _ = cv2.Rodrigues(r0)
    t0 = t0.reshape(3,1)

    # ✅ Coordenadas 2D reales en el plano
    puntos_2D = {}
    centros_pix = {}

    for i, marker_id in enumerate(ids):
        rvec, tvec = poses[marker_id]
        t = tvec.reshape(3,1)
        relative = R0.T @ (t - t0)

        x = float(relative[0])
        y = float(relative[1])
        puntos_2D[marker_id] = (x, y)

        # centro en pixeles
        cx = int(np.mean(corners[i][0][:,0]))
        cy = int(np.mean(corners[i][0][:,1]))
        centros_pix[marker_id] = (cx, cy)

    # ✅ ordenar para polígono
    pts = np.array(list(puntos_2D.values()))
    centro = np.mean(pts, axis=0)
    ang = np.arctan2(pts[:,1]-centro[1], pts[:,0]-centro[0])
    orden = np.argsort(ang)
    ids_ordenados = np.array(list(puntos_2D.keys()))[orden]

    # ✅ dibujar polígono
    pix = [centros_pix[i] for i in ids_ordenados]
    for i in range(len(pix)):
        p1 = pix[i]
        p2 = pix[(i+1)%len(pix)]
        cv2.line(frame, p1, p2, (255, 0, 0), 4)

    # ✅ área en centímetros
    area_pts = np.array([puntos_2D[i] for i in ids_ordenados])

    return puntos_2D, area_pts, frame, None


In [79]:
frame = cv2.imread(image_path)

coords_2D, area_pts, img_out, error = detectar_area_aruco(
    frame,
    marker_size_cm=2.7,
    origin_id=0
)

if error:
    print(error)
else:
    from google.colab.patches import cv2_imshow
    cv2_imshow(img_out)

    print("\nCoordenadas 2D reales (cm):")
    for k,(x,y) in coords_2D.items():
        print(f"ID {k}: X={x:.2f} cm, Y={y:.2f} cm")

    print("\nÁrea (cm):")
    print(area_pts)
